# Apply trained models onto respective failing cells.

In [1]:
import pathlib
import sys

import joblib
import numpy as np
import pandas as pd
from joblib import load
from plotnine import (
    aes,
    element_text,
    facet_grid,
    facet_wrap,
    geom_density,
    ggplot,
    labs,
    scale_color_manual,
    scale_fill_manual,
    scale_x_continuous,
    theme,
    theme_bw,
)
from plotnine.options import set_option

sys.path.append(str(pathlib.Path.cwd().resolve().parent / "figure_3"))
from figure3_utils import get_X_y_data


## Helper functions

In [2]:
def per_class_accuracy(
    y_true: np.ndarray, y_pred: np.ndarray, model_name: str
) -> list[dict]:
    """Compute accuracy separately for class 0 (failing) and class 1 (healthy).

    Args:
        y_true (np.ndarray): True labels.
        y_pred (np.ndarray): Predicted labels.
        model_name (str): Name of the model.

    Returns:
        list[dict]: List of dictionaries containing model name, class label, accuracy,
        and number of samples for each class.
    """
    results = []
    for cls, cls_label in [(0, "failing"), (1, "healthy")]:
        mask = y_true == cls
        acc = (y_pred[mask] == y_true[mask]).mean()
        results.append(
            {
                "model": model_name,
                "class": cls_label,
                "accuracy": acc,
                "n": mask.sum(),
            }
        )
    return results


In [3]:
def apply_model_to_group(
    df: pd.DataFrame,
    feature_cols: list[str],
    model: joblib.Parallel,
    group_label: str,
    model_label: str,
) -> pd.DataFrame:
    """Score a failing-cell group with a model and label the result.

    Args:
        df (pd.DataFrame): Failing-cell group (with "Metadata_cell_type" and
            `feature_cols`).
        feature_cols (list[str]): Feature columns `model` was trained on, in order.
        model (joblib.Parallel): Fitted classifier with `predict_proba`.
        group_label (str): Name of the failing-cell group (e.g. "Both").
        model_label (str): Name of the model used to score the group.

    Returns:
        pd.DataFrame: Predicted probability, true class, group, and model per cell.
    """
    filtered_df = df[metadata_cols_to_keep + feature_cols].dropna(subset=feature_cols)
    X, y = get_X_y_data(df=filtered_df, label="Metadata_cell_type")
    y_binary = le.transform(y)
    y_probs = model.predict_proba(X)[:, 1]
    return pd.DataFrame(
        {
            "predicted_prob": y_probs,
            "true_class": np.where(y_binary == 0, "Diseased", "Healthy"),
            "group": group_label,
            "model": model_label,
        }
    )


In [4]:
figure_path = pathlib.Path("./figures")
# make directory if it doesn't already exist
figure_path.mkdir(exist_ok=True)


## Load in label encoder

In [5]:
# load in label encoder
le = load(
    pathlib.Path(
        "/media/18tbdrive/1.Github_Repositories/cellpainting_predicts_cardiac_fibrosis/5.machine_learning/0.train_logistic_regression/encoder_results/label_encoder_log_reg_fs_plate_4.joblib"
    )
)


/home/jenna/.cache/pypoetry/virtualenvs/cosmicqc-vNopUmqk-py3.11/lib/python3.11/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.3.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations


## Load in model

In [6]:
# Load the trained models
no_QC_model = joblib.load(
    pathlib.Path(
        "/media/18tbdrive/1.Github_Repositories/cellpainting_predicts_cardiac_fibrosis/5.machine_learning/0.train_logistic_regression/models/no_QC_models/log_reg_fs_plate_4_final_downsample_no_QC.joblib"
    )
)

coSMicQC_model = joblib.load(
    pathlib.Path(
        "/media/18tbdrive/1.Github_Repositories/cellpainting_predicts_cardiac_fibrosis/5.machine_learning/0.train_logistic_regression/models/log_reg_fs_plate_4_final_downsample.joblib"
    )
)

ECOD_model = joblib.load(
    pathlib.Path(
        "../figure_3/models/log_reg_fs_ecod_final_downsample.joblib"
    )
)


/home/jenna/.cache/pypoetry/virtualenvs/cosmicqc-vNopUmqk-py3.11/lib/python3.11/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.3.2 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
/home/jenna/.cache/pypoetry/virtualenvs/cosmicqc-vNopUmqk-py3.11/lib/python3.11/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.9.0 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations


## Load in no-QC dataset and filter for failing cells

In [ ]:
# Load the normalized QC plate 4 to be able to filter features
plate_4_no_QC = pd.read_parquet(
    pathlib.Path(
        "/media/18tbdrive/1.Github_Repositories/cellpainting_predicts_cardiac_fibrosis/3.process_cfret_features/data/single_cell_profiles/localhost231120090001_sc_normalized_no_QC.parquet"
    )
)

# Load the failing cells metadata (contains per-cell QC failure booleans)
failing_cells_metadata = pd.read_parquet(
    pathlib.Path("../figure_3/failing_cells_metadata/failing_cells_metadata.parquet")
)

# Merge plate_4_no_QC with the QC failure metadata on shared cell identifiers
merge_cols = [
    "Metadata_Well",
    "Metadata_Site",
    "Metadata_Nuclei_Location_Center_X",
    "Metadata_Nuclei_Location_Center_Y",
]
plate_4_no_QC_merged = plate_4_no_QC.merge(
    failing_cells_metadata, on=merge_cols, how="inner"
)

# Filter into separate dataframes for each QC failure type
ecod_df = plate_4_no_QC_merged[plate_4_no_QC_merged["failed_pyod_ecod"]]

cosmicqc_df = plate_4_no_QC_merged[plate_4_no_QC_merged["failed_cosmicqc"]]

both_df = plate_4_no_QC_merged[plate_4_no_QC_merged["failed_both"]]

# Cells failing only one method (i.e. excluding the overlap with the other method)
cosmicqc_only_df = plate_4_no_QC_merged[
    plate_4_no_QC_merged["failed_cosmicqc"] & ~plate_4_no_QC_merged["failed_both"]
]
ecod_only_df = plate_4_no_QC_merged[
    plate_4_no_QC_merged["failed_pyod_ecod"] & ~plate_4_no_QC_merged["failed_both"]
]

print(f"ECOD (all failures, including overlap with coSMicQC): {ecod_df.shape[0]} cells")
print(f"coSMicQC (all failures, including overlap with ECOD): {cosmicqc_df.shape[0]} cells")
print(f"Both: {both_df.shape[0]} cells")
print(f"coSMicQC only (excluding overlap with ECOD): {cosmicqc_only_df.shape[0]} cells")
print(f"ECOD only (excluding overlap with coSMicQC): {ecod_only_df.shape[0]} cells")


ECOD (all failures, including overlap with coSMicQC): 4005 cells
coSMicQC (all failures, including overlap with ECOD): 3996 cells
Both: 2191 cells
coSMicQC only (excluding overlap with ECOD): 1805 cells
ECOD only (excluding overlap with coSMicQC): 1814 cells


## Load in feature selected profiles from coSMicQC and ECOD model training to collect features

In [8]:
# Get feature columns (excluding any Metadata_ columns) used by each model's training data
coSMicQC_plate_4_features = pd.read_parquet(
    pathlib.Path(
        "/media/18tbdrive/1.Github_Repositories/cellpainting_predicts_cardiac_fibrosis/3.process_cfret_features/data/single_cell_profiles/localhost231120090001_sc_feature_selected.parquet"
    )
)
ECOD_plate_4_features = pd.read_parquet(
    pathlib.Path("../figure_3/models/idc_normalized_feature_selected.parquet")
)
no_QC_plate_4_features = pd.read_parquet(
    pathlib.Path(
        "/media/18tbdrive/1.Github_Repositories/cellpainting_predicts_cardiac_fibrosis/3.process_cfret_features/data/single_cell_profiles/localhost231120090001_sc_feature_selected_no_QC.parquet"
    )
)

cosmicqc_feature_cols = [
    col for col in coSMicQC_plate_4_features.columns if not col.startswith("Metadata_")
]
ecod_feature_cols = [
    col for col in ECOD_plate_4_features.columns if not col.startswith("Metadata_")
]
no_qc_feature_cols = [
    col for col in no_QC_plate_4_features.columns if not col.startswith("Metadata_")
]

print(f"coSMicQC feature columns: {len(cosmicqc_feature_cols)}")
print(f"ECOD feature columns: {len(ecod_feature_cols)}")
print(f"No QC feature columns: {len(no_qc_feature_cols)}")


coSMicQC feature columns: 625
ECOD feature columns: 701
No QC feature columns: 648


In [9]:
# Metadata columns needed downstream (label column for get_X_y_data)
metadata_cols_to_keep = ["Metadata_cell_type"]

# Filter each dataset down to only the columns its respective model was trained on
cosmicqc_filtered_df = cosmicqc_df[metadata_cols_to_keep + cosmicqc_feature_cols]
ecod_filtered_df = ecod_df[metadata_cols_to_keep + ecod_feature_cols]

# Drop rows with any NaN in the feature columns before predicting
cosmicqc_before = cosmicqc_filtered_df.shape[0]
ecod_before = ecod_filtered_df.shape[0]

cosmicqc_filtered_df = cosmicqc_filtered_df.dropna(subset=cosmicqc_feature_cols)
ecod_filtered_df = ecod_filtered_df.dropna(subset=ecod_feature_cols)

print(
    f"coSMicQC rows after dropping NaNs: {cosmicqc_filtered_df.shape[0]} (dropped {cosmicqc_before - cosmicqc_filtered_df.shape[0]})"
)
print(
    f"ECOD rows after dropping NaNs: {ecod_filtered_df.shape[0]} (dropped {ecod_before - ecod_filtered_df.shape[0]})"
)

# Load in X and y data for the coSMicQC-filtered dataset
X_cosmicqc, y_cosmicqc = get_X_y_data(
    df=cosmicqc_filtered_df, label="Metadata_cell_type"
)
y_binary_cosmicqc = le.transform(y_cosmicqc)
y_probs_cosmicqc = coSMicQC_model.predict_proba(X_cosmicqc)[:, 1]

# Load in X and y data for the ECOD-filtered dataset
X_ecod, y_ecod = get_X_y_data(df=ecod_filtered_df, label="Metadata_cell_type")
y_binary_ecod = le.transform(y_ecod)
y_probs_ecod = ECOD_model.predict_proba(X_ecod)[:, 1]


coSMicQC rows after dropping NaNs: 3993 (dropped 3)
ECOD rows after dropping NaNs: 3999 (dropped 6)


In [10]:
print(f"coSMicQC positive class prevalence: {y_binary_cosmicqc.mean():.3f}")
print(f"ECOD positive class prevalence: {y_binary_ecod.mean():.3f}")


coSMicQC positive class prevalence: 0.391
ECOD positive class prevalence: 0.388


In [11]:
# Convert probabilities to predicted class labels using a 0.5 threshold
threshold = 0.5
y_pred_cosmicqc = (y_probs_cosmicqc >= threshold).astype(int)
y_pred_ecod = (y_probs_ecod >= threshold).astype(int)

acc_records = (
    per_class_accuracy(y_binary_cosmicqc, y_pred_cosmicqc, "coSMicQC")
    + per_class_accuracy(y_binary_ecod, y_pred_ecod, "ECOD")
)
acc_df = pd.DataFrame(acc_records)
acc_df


,model,class,accuracy,n
0,coSMicQC,failing,0.403783,2432
1,coSMicQC,healthy,0.937220,1561
2,ECOD,failing,0.665033,2448
3,ECOD,healthy,0.827208,1551


## Distribution of predicted probabilities on failing cells, by QC model and true class

Apply all three models (no-QC, coSMicQC, ECOD) to each failing-cell group — cells
caught by coSMicQC only, by ECOD only, and by both methods — and plot the resulting
predicted probabilities, faceted by which model produced the prediction (columns)
and the cell's true class (rows), with one density line per failing-cell group
within each facet. This shows how well-separated the two classes are according to
each model, within each QC method's failing-cell population.

In [12]:
# Failing-cell groups (from each QC method's failures) to score
failing_cell_groups = {
    "coSMicQC only": cosmicqc_only_df,
    "PyOD ECOD only": ecod_only_df,
    "Both": both_df,
}

# Models to apply to each group, each with its own feature set
models_to_apply = {
    "No QC": (no_QC_model, no_qc_feature_cols),
    "coSMicQC": (coSMicQC_model, cosmicqc_feature_cols),
    "ECOD": (ECOD_model, ecod_feature_cols),
}

# Apply every model to every failing-cell group
df_failing_probs = pd.concat(
    [
        apply_model_to_group(
            df=group_df,
            feature_cols=feature_cols,
            model=model,
            group_label=group_label,
            model_label=model_label,
        )
        for group_label, group_df in failing_cell_groups.items()
        for model_label, (model, feature_cols) in models_to_apply.items()
    ]
)

# Fix facet column order (No QC, coSMicQC, ECOD) rather than alphabetical
df_failing_probs["model"] = pd.Categorical(
    df_failing_probs["model"], categories=["No QC", "coSMicQC", "ECOD"]
)

df_failing_probs.groupby(["model", "group", "true_class"])["predicted_prob"].describe()


/tmp/ipykernel_1873595/2501845231.py:35: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.


count      mean       std           min  \
model    group          true_class                                             
No QC    Both           Diseased    1240.0  0.281609  0.310209  1.183983e-09   
                        Healthy      951.0  0.796711  0.264558  1.637489e-03   
         PyOD ECOD only Diseased    1210.0  0.256232  0.298100  3.898055e-08   
                        Healthy      604.0  0.817203  0.246604  1.789878e-03   
         coSMicQC only  Diseased    1194.0  0.209552  0.255280  2.137833e-05   
                        Healthy      611.0  0.786371  0.247901  5.668227e-02   
coSMicQC Both           Diseased    1238.0  0.568404  0.355696  2.230521e-06   
                        Healthy      950.0  0.887339  0.215613  1.964064e-03   
         PyOD ECOD only Diseased    1210.0  0.234291  0.271696  1.334983e-06   
                        Healthy      604.0  0.778018  0.261658  1.482341e-02   
         coSMicQC only  Diseased    1194.0  0.587621  0.324433  1.458492e-04   
                        Healthy      611.0  0.927047  0.142337  3.959358e-03   
ECOD     Both           Diseased    1238.0  0.318133  0.295964  2.964399e-05   
                        Healthy      950.0  0.751539  0.279937  7.257245e-05   
         PyOD ECOD only Diseased    1210.0  0.423148  0.305597  6.569865e-05   
                        Healthy      601.0  0.813928  0.246250  1.957590e-03   
         coSMicQC only  Diseased    1194.0  0.289880  0.251514  1.696454e-04   
                        Healthy      611.0  0.807352  0.202571  4.006121e-02   

                                         25%       50%       75%       max  
model    group          true_class                                          
No QC    Both           Diseased    0.029877  0.138377  0.475292  0.999931  
                        Healthy     0.678004  0.933639  0.992007  1.000000  
         PyOD ECOD only Diseased    0.026155  0.118617  0.408721  1.000000  
                        Healthy     0.718608  0.937573  0.991416  1.000000  
         coSMicQC only  Diseased    0.025971  0.091075  0.302638  0.999700  
                        Healthy     0.665750  0.892353  0.980325  0.999993  
coSMicQC Both           Diseased    0.210904  0.643062  0.916180  0.999993  
                        Healthy     0.910395  0.985759  0.998000  1.000000  
         PyOD ECOD only Diseased    0.029925  0.112806  0.351056  1.000000  
                        Healthy     0.649220  0.896095  0.976753  0.999997  
         coSMicQC only  Diseased    0.294897  0.652780  0.890915  0.999985  
                        Healthy     0.935683  0.982471  0.996457  0.999990  
ECOD     Both           Diseased    0.062627  0.211621  0.536199  0.998231  
                        Healthy     0.617339  0.876124  0.968374  0.999986  
         PyOD ECOD only Diseased    0.136057  0.386431  0.691004  0.999994  
                        Healthy     0.737602  0.933647  0.980257  0.999995  
         coSMicQC only  Diseased    0.085629  0.205635  0.447131  0.991544  
                        Healthy     0.730763  0.875656  0.959624  0.999961

In [13]:
# Count how many samples of each true class are in each failing-cell group,
# per model (NaN-dropping is model-specific, so counts can vary slightly)
class_counts_by_group = (
    df_failing_probs.groupby(["model", "group", "true_class"], observed=True)
    .size()
    .unstack(fill_value=0)
)

class_counts_by_group


true_class               Diseased  Healthy
model    group                            
No QC    Both                1240      951
         PyOD ECOD only      1210      604
         coSMicQC only       1194      611
coSMicQC Both                1238      950
         PyOD ECOD only      1210      604
         coSMicQC only       1194      611
ECOD     Both                1238      950
         PyOD ECOD only      1210      601
         coSMicQC only       1194      611

In [14]:
# Set the figure size (update as needed) -- wider to fit 3 model columns
height = 14
width = 26
set_option("figure_size", (width, height))

# Plot: predicted probability distributions on failing cells, faceted by QC
# model (columns) and true class (rows), colored by failing-cell group
group_colors = {
    "coSMicQC only": "#CC79A7",
    "PyOD ECOD only": "#0072B2",
    "Both": "#D55E00",
}

# Add prefixed label columns so facet strips read "True class: X" / "Model: Y"
df_failing_probs = df_failing_probs.assign(
    true_class_label=lambda d: "True class: " + d["true_class"].astype(str),
    model_label=lambda d: "Model: " + d["model"].astype(str),
)

failing_probs_plot = (
    ggplot(df_failing_probs, aes(x="predicted_prob", color="group", fill="group"))
    + geom_density(size=1.2, alpha=0.3)
    + facet_wrap("~ true_class_label + model_label", scales="free_y")
    + scale_color_manual(values=group_colors)
    + scale_fill_manual(values=group_colors)
    + scale_x_continuous(labels=lambda breaks: [f"{b:g}" for b in breaks])
    + labs(
        x="Predicted probability (healthy class = 1)",
        y="Density",
        color="QC failure status",
        fill="QC failure status",
    )
    + theme_bw()
    + theme(
        legend_position="right",
        axis_title=element_text(size=38),
        axis_text=element_text(size=34),
        legend_title=element_text(size=36),
        legend_text=element_text(size=34),
        strip_text=element_text(size=36),
    )
)

failing_probs_plot.save(
    f"{figure_path}/predicted_probability_distributions_failing_cells.png",
    dpi=600,
    limitsize=False
)


/home/jenna/.cache/pypoetry/virtualenvs/cosmicqc-vNopUmqk-py3.11/lib/python3.11/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 26 x 14 in image.
/home/jenna/.cache/pypoetry/virtualenvs/cosmicqc-vNopUmqk-py3.11/lib/python3.11/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: figures/predicted_probability_distributions_failing_cells.png
